# Ollama Raw LLM Completion


## Setup

Run Ollama before executing cells:
- `ollama serve`
- `ollama pull gemma3:4b` (or another <=12B local model)

## Ollama API

In [ ]:
import requests
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import threading

# API Setting
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:4b"

app = FastAPI(title="HKBU Study Companion")

# Query Format
class Query(BaseModel):
    question: str
    context: str = "None"
    is_search: bool
    use_neural_retrieval: bool

def build_prompt(context: str, question: str):
    return f"""
You are a helpful HKBU study assistant.
Answer the question based ONLY on the provided context.
If you don't know, say "I don't have enough information."

Context:
{context}

Question:
{question}

Answer (concise with citation):
"""

def complete_document(prefix: str):
    payload = {
        "model": MODEL,
        "prompt": prefix,
        "stream": False,
        "raw": True,
        "options": {
            "num_predict": 256,
            "temperature": 0.3
        }
    }
    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=120)
        r.raise_for_status()
        return r.json()["response"]
    except Exception as e:
        return f"Error: {str(e)}"

# API
@app.post("/ask")
def ask(q: Query):
    prompt = build_prompt(q.context, q.question)
    answer = complete_document(prompt)
    return {
        "question": q.question,
        "context": q.context,
        "answer": answer
    }

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("✅ Backend Address: http://localhost:8000")
print("✅ Backend Document: http://localhost:8000/docs")

✅ Backend Address: http://localhost:8000
✅ Backend Document: http://localhost:8000/docs


INFO:     Started server process [27448]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 8000): 通常每个套接字地址(协议/网络地址/端口)只允许使用一次。
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:63335 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:63335 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:50974 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:50974 - "GET /openapi.json HTTP/1.1" 200 OK


If you get `ConnectionError`, start Ollama first (`ollama serve`) and ensure your model is available (`ollama pull llama3.1:8b`).
If `requests` is missing, run `%pip install requests` in a notebook cell.
